# Lesson 05: Model Fitting (Hough Transform, RANSAC) — Python (Colab)

This notebook is split into explained segments showing how to fit a line to noisy data with outliers using Hough voting and RANSAC consensus.


## 1. Setup and generate noisy data with outliers

Imports libraries and creates a dataset: many inlier points near a true line plus outlier points. Robust fitting aims to recover the line despite outliers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate points: inliers on a line + outliers
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y_true = 0.6*x + 1.5
y_in = y_true + rng.normal(0, 0.25, size=x.size)
x_out = rng.uniform(0,10,12); y_out = rng.uniform(0,8,12)
X = np.concatenate([x, x_out]); Y = np.concatenate([y_in, y_out])


## 2. Hough transform (parameter-space voting)

Parameterizes lines as `ρ = x cosθ + y sinθ`. Each point votes across θ values in an accumulator; the peak corresponds to the line supported by the most points.


In [ ]:
# --- Hough transform (rho = x cosθ + y sinθ) ---
thetas = np.deg2rad(np.arange(-90, 90))
diag = int(np.ceil(np.hypot(X.max()-X.min(), Y.max()-Y.min()))) + 10
rhos = np.arange(-diag, diag+1)
acc = np.zeros((len(rhos), len(thetas)), dtype=np.int32)
for xi, yi in zip(X,Y):
    rho_vals = (xi*np.cos(thetas) + yi*np.sin(thetas)).astype(int)
    for ti, r in enumerate(rho_vals):
        ri = r + diag
        if 0 <= ri < len(rhos): acc[ri, ti] += 1
peak = np.unravel_index(np.argmax(acc), acc.shape)
rho_p, theta_p = rhos[peak[0]], thetas[peak[1]]


## 3. RANSAC: hypothesis, inlier counting, and refit

Implements RANSAC for line fitting: repeatedly sample 2 points, fit a candidate line, count inliers within a distance threshold, keep the best consensus, then refit using all inliers for a better estimate.


In [ ]:
# --- RANSAC line fit (y = m*x + c form for demo) ---
def fit_line(p1, p2):
    (x1,y1),(x2,y2)=p1,p2
    if abs(x2-x1) < 1e-9: return None
    m=(y2-y1)/(x2-x1); c=y1-m*x1; return m,c

def ransac(X,Y, iters=200, thresh=0.35):
    best=None; best_inliers=None
    pts=list(zip(X,Y))
    for _ in range(iters):
        i,j = rng.choice(len(pts), 2, replace=False)
        model = fit_line(pts[i], pts[j])
        if model is None: continue
        m,c = model
        err = np.abs(m*X - Y + c)/np.sqrt(m*m+1)
        inliers = err < thresh
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best, best_inliers = model, inliers
    # refit on inliers
    Xi,Yi = X[best_inliers], Y[best_inliers]
    A = np.c_[Xi, np.ones_like(Xi)]
    m,c = np.linalg.lstsq(A,Yi,rcond=None)[0]
    return (m,c), best_inliers

(mr,cr), inliers = ransac(X,Y)


## 4. Visualize Hough accumulator and RANSAC result

Shows the accumulator with its peak and plots inliers/outliers with the robustly fitted line to compare both approaches.


In [ ]:
fig,axs = plt.subplots(1,2,figsize=(12,4.5))
axs[0].imshow(acc, cmap='hot', aspect='auto', extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[0], rhos[-1]])
axs[0].set_title('Hough accumulator (peak marked)')
axs[0].plot(np.rad2deg(theta_p), rho_p, 'c+', ms=12)
axs[1].scatter(X[~inliers], Y[~inliers], c='gray', label='outliers')
axs[1].scatter(X[inliers], Y[inliers], c='#1f5f7a', label='inliers')
xs = np.linspace(0,10,100)
axs[1].plot(xs, mr*xs+cr, 'r', lw=2, label='RANSAC fit')
axs[1].legend(); axs[1].set_title('RANSAC line fitting')
plt.tight_layout(); plt.show()
print('Hough peak theta(deg)=%.1f rho=%.2f' % (np.rad2deg(theta_p), rho_p))
print('RANSAC inliers: %d/%d  model: y=%.3fx+%.3f' % (inliers.sum(), len(X), mr, cr))


## Experiments

- Detect multiple lines by suppressing peaks iteratively, fit circles (3D accumulator or RANSAC), or use `cv2.HoughLines` and homography RANSAC on feature matches.
- Vary outlier ratio, threshold, and iteration count to see effects on robustness.
